Undersampling + Residual BLock + Sliding Window CNN + LSTM (no Attention)

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [10]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [10]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giảm từ 49200 xuống 20000 cửa sổ.
Class 1: Giảm từ 122644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 18432 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 110569 xuống 20000 cửa sổ.
Class 4: Giảm từ 55098 xuống 20000 cửa sổ.
Class 5: Giảm từ 46800 xuống 20000 cửa sổ.
Class 6: Giảm từ 137896 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 13810 cửa sổ (step=5).
Class 8: Giảm từ 31452 xuống 20000 cửa sổ.
Class 9: Giữ nguyên 392 cửa sổ (step=5).
Class 10: Giữ nguyên 270 cửa sổ (step=5).
Class 11: Giữ nguyên 5194 cửa sổ (step=5).
Class 12: Giảm từ 67231 xuống 20000 cửa sổ.
Tổng số cửa sổ sau khi lọc và Undersampling: 198098
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 328000 cửa sổ (step=1).
Class 1: Giữ nguyên 817622 cửa sổ (step=

In [4]:
import torch
import torch.nn as nn

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# MODEL CNN-BiLSTM (ĐÃ LƯỢC BỎ ATTENTION)
class CNN_BiLSTM_NoAttention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_NoAttention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Đã loại bỏ self.attention
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        # THAY THẾ ATTENTION: Sử dụng Global Average Pooling theo chiều sequence
        # Lấy trung bình tất cả các time-steps để nén về (Batch, hidden_size * 2)
        context_vector = torch.mean(out, dim=1)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [5]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7374, Train Macro F1: 0.8465 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6247, Train Macro F1: 0.8964 | Val Loss: 0.5439, Val Macro F1: 0.9015


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6217, Train Macro F1: 0.9001 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6190, Train Macro F1: 0.8990 | Val Loss: 0.5419, Val Macro F1: 0.9017


...

Epoch [28/30] - Train Loss: 0.6096, Train Macro F1: 0.9001 | Val Loss: 0.5407, Val Macro F1: 0.9021


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6098, Train Macro F1: 0.8998 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6097, Train Macro F1: 0.8998 | Val Loss: 0.5406, Val Macro F1: 0.9022

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

              precision    recall  f1-score   support

           0     1.0000    0.9995    0.9997    246000
           1     0.9928    1.0000    0.9964    613218
           2     0.9894    0.8641    0.9225     92157
           3     1.0000    0.9999    1.0000    552847
           4     1.0000    0.9813    0.9905    275494
           5     1.0000    1.0000    1.0000    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9999    0.8274    0.9055     69052
           8     0.9972    1.0000    0.9986    157261
           9     0.5922    0.9939    0.7422      1962
          10     0.0000    0.0000    0.0000      1351
          11     0.5052    0.9726    0.6649     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9903   3294954
   macro avg     0.8520    0.8953    0.8631   3294954
weighted avg     0.9937    0.9903    0.9911   3294954



In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7305, Train Macro F1: 0.8481 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6226, Train Macro F1: 0.9065 | Val Loss: 0.5441, Val Macro F1: 0.9008


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6187, Train Macro F1: 0.8988 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6173, Train Macro F1: 0.8996 | Val Loss: 0.5428, Val Macro F1: 0.9016


...

Epoch [28/30] - Train Loss: 0.6086, Train Macro F1: 0.9006 | Val Loss: 0.5409, Val Macro F1: 0.9020


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6086, Train Macro F1: 0.8999 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6086, Train Macro F1: 0.9004 | Val Loss: 0.5409, Val Macro F1: 0.9020

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

              precision    recall  f1-score   support

           0     1.0000    0.9985    0.9993    246000
           1     0.9994    1.0000    0.9997    613218
           2     0.8491    0.8450    0.8471     92157
           3     1.0000    0.9999    1.0000    552847
           4     1.0000    0.9830    0.9915    275494
           5     0.9996    1.0000    0.9998    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9997    0.8542    0.9213     69052
           8     0.9998    1.0000    0.9999    157261
           9     0.5829    0.9888    0.7335      1962
          10     0.0000    0.0000    0.0000      1351
          11     0.6099    0.9638    0.7471     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9904   3294954
   macro avg     0.8493    0.8949    0.8645   3294954
weighted avg     0.9919    0.9904    0.9907   3294954



UnderSampling + Residual Block + CNN + LSTM + Attention (no Sliding Window)

In [7]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        if x.size(-1) >=2:
            x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [9]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledDataset(Dataset):
    def __init__(self, df, max_samples_per_class=20000):
        # Trích xuất Features và Labels
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        
        print(f"Đang tính toán các mẫu hợp lệ và Undersampling...")
        
        # Mảng chứa tất cả index từ 0 đến len(df) - 1
        all_indices = np.arange(len(self.X))
        all_labels = self.y.numpy()
        
        self.valid_indices = []
        
        # Lặp qua từng class
        classes = np.unique(all_labels)
        rng = np.random.default_rng(seed=42)
        
        for c in classes:
            # Lấy tất cả index của các mẫu thuộc class c
            c_indices = all_indices[all_labels == c]
            count = len(c_indices)
            
            # Nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # Chọn ngẫu nhiên không hoàn lại
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} mẫu.")
            else:
                # Nếu ít hơn hoặc bằng ngưỡng thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} mẫu.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn khi train
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số mẫu sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy index thực sự đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # Lấy feature và label tại dòng (index) tương ứng
        sample_X = self.X[actual_idx].unsqueeze(0) 
        label_y = self.y[actual_idx]
        
        return sample_X, label_y

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).")
train_dataset = UndersampledDataset(train_df, max_samples_per_class=MAX_SAMPLES)


print(f"Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledDataset(valid_df, max_samples_per_class=10000000)
test_dataset = UndersampledDataset(test_df, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giảm từ 246000 xuống 20000 mẫu.
Class 1: Giảm từ 613217 xuống 20000 mẫu.
Class 2: Giảm từ 92156 xuống 20000 mẫu.
Class 3: Giảm từ 552845 xuống 20000 mẫu.
Class 4: Giảm từ 275492 xuống 20000 mẫu.
Class 5: Giảm từ 234000 xuống 20000 mẫu.
Class 6: Giảm từ 689482 xuống 20000 mẫu.
Class 7: Giảm từ 69051 xuống 20000 mẫu.
Class 8: Giảm từ 157260 xuống 20000 mẫu.
Class 9: Giữ nguyên 1960 mẫu.
Class 10: Giữ nguyên 1358 mẫu.
Class 11: Giảm từ 25971 xuống 20000 mẫu.
Class 12: Giảm từ 336153 xuống 20000 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 223318
Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 328000 mẫu.
Class 1: Giữ nguyên 817622 mẫu.
Class 2: Giữ nguyên 122874 mẫu.
Class 3: Giữ nguyên 737127 mẫu.
Class 4: Giữ nguyên 367323 mẫu.
Class 5: Giữ nguyên 31200

In [11]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.8001, Train Macro F1: 0.9013 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6564, Train Macro F1: 0.9821 | Val Loss: 0.5565, Val Macro F1: 0.9765


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6469, Train Macro F1: 0.9834 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6430, Train Macro F1: 0.9839 | Val Loss: 0.5559, Val Macro F1: 0.9259


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.6240, Train Macro F1: 0.9897 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.6232, Train Macro F1: 0.9891 | Val Loss: 0.5476, Val Macro F1: 0.9778


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6228, Train Macro F1: 0.9891 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6229, Train Macro F1: 0.9897 | Val Loss: 0.5485, Val Macro F1: 0.9829

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9936    0.9990    0.9963    246000
           1     1.0000    0.9919    0.9959    613218
           2     0.8687    0.8136    0.8402     92157
           3     0.9999    0.9969    0.9984    552847
           4     0.9971    0.8751    0.9321    275494
           5     0.9983    0.9970    0.9977    234000
           6     1.0000    0.9994    0.9997    689483
           7     0.9543    0.6582    0.7790     69052
           8     0.8168    0.9999    0.8991    157261
           9     0.9311    0.8262    0.8755      1962
          10     0.8441    0.9713    0.9032      1360
          11     0.4244    0.9419    0.5851     25973
          12     0.9999    1.0000    0.9999    336156

    accuracy                         0.9742   3294963
   macro avg     0.9099    0.9285    0.9079   3294963
weighted avg     0.9811    0.9742    0.9755   3294963



In [12]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1/30 [Train]:   0%|          | 0/872 [00:00<?, ?it/s]

Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7828, Train Macro F1: 0.8969 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6568, Train Macro F1: 0.9810 | Val Loss: 0.5529, Val Macro F1: 0.9769


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6486, Train Macro F1: 0.9842 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6448, Train Macro F1: 0.9840 | Val Loss: 0.5536, Val Macro F1: 0.9809


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.6244, Train Macro F1: 0.9898 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.6248, Train Macro F1: 0.9901 | Val Loss: 0.5518, Val Macro F1: 0.9734


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6248, Train Macro F1: 0.9901 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6246, Train Macro F1: 0.9897 | Val Loss: 0.5509, Val Macro F1: 0.9707

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9934    0.9990    0.9962    246000
           1     0.9991    0.9974    0.9983    613218
           2     0.8931    0.9152    0.9040     92157
           3     0.9999    0.9969    0.9984    552847
           4     0.9997    0.9065    0.9508    275494
           5     0.9973    0.9998    0.9986    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9992    0.6123    0.7593     69052
           8     0.8634    0.9989    0.9262    157261
           9     0.9963    0.8272    0.9039      1962
          10     0.7060    0.9904    0.8244      1360
          11     0.4710    0.9430    0.6283     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9800   3294963
   macro avg     0.9168    0.9374    0.9145   3294963
weighted avg     0.9853    0.9800    0.9806   3294963



Dynamic Undersampling + Residual Block + CNN + LSTM + Attention + Sliding Window

In [13]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [15]:
class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False, rare_classes=None):
        

        if rare_classes is None:
            rare_classes = [9, 10]
            
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        # Tạo 2 mảng index: 1 cái step=1 (bảo toàn), 1 cái có step (băm mỏng)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels_step1):
            if c in rare_classes:
                # Bảo toàn 100% dữ liệu cho class hiếm
                self.class_indices[c] = all_indices_step1[window_labels_step1 == c]
            else:
                # Băm mỏng theo step đối với các class đa số
                self.class_indices[c] = all_indices_stepped[window_labels_stepped == c]
            
            print(f"Class {c}: Có sẵn {len(self.class_indices[c])} cửa sổ trong Pool")
            
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            
            # NẾU LÀ TẬP TRAIN (có bật resample_each_epoch)
            if self.resample_each_epoch:
                if count > self.max_samples_per_class:
                    # Dư thì băm đi (Undersampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=False)
                else:
                    # Thiếu thì nhân bản (Oversampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=True)
                    
            # NẾU LÀ TẬP VAL/TEST (Giữ nguyên gốc 100%, không thêm không bớt)
            else:
                sampled = c_indices
                
            self.valid_indices.extend(sampled)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Dynamicsampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = DynamicUndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE, resample_each_epoch=True)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Dynamicsampling, set Train Step = 5)...
Class 0: Có sẵn 49200 cửa sổ trong Pool
Class 1: Có sẵn 122644 cửa sổ trong Pool
Class 2: Có sẵn 18432 cửa sổ trong Pool
Class 3: Có sẵn 110569 cửa sổ trong Pool
Class 4: Có sẵn 55098 cửa sổ trong Pool
Class 5: Có sẵn 46800 cửa sổ trong Pool
Class 6: Có sẵn 137896 cửa sổ trong Pool
Class 7: Có sẵn 13810 cửa sổ trong Pool
Class 8: Có sẵn 31452 cửa sổ trong Pool
Class 9: Có sẵn 1960 cửa sổ trong Pool
Class 10: Có sẵn 1349 cửa sổ trong Pool
Class 11: Có sẵn 5194 cửa sổ trong Pool
Class 12: Có sẵn 67231 cửa sổ trong Pool
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 328000 cửa sổ (step=1).
Class 1: Giữ nguyên 817622 cửa sổ (step=1).
Class 2: Giữ nguyên 122874 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 737127 cửa sổ (step=1).
Class 4: Giữ nguyên 367323 cửa sổ (step=1).
Class 5: Giữ nguyên 

In [17]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start() 
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7116, Train Macro F1: 0.9770 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6257, Train Macro F1: 0.9983 | Val Loss: 0.5427, Val Macro F1: 0.9967


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6239, Train Macro F1: 0.9985 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6215, Train Macro F1: 0.9988 | Val Loss: 0.5419, Val Macro F1: 0.9960


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.6166, Train Macro F1: 0.9993 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.6158, Train Macro F1: 0.9995 | Val Loss: 0.5403, Val Macro F1: 0.9966


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6153, Train Macro F1: 0.9995 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6152, Train Macro F1: 0.9996 | Val Loss: 0.5403, Val Macro F1: 0.9962

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9999    0.9996    0.9998    246000
           1     0.9950    0.9997    0.9974    613218
           2     0.9827    0.8344    0.9025     92157
           3     1.0000    0.9999    0.9999    552847
           4     1.0000    0.9738    0.9867    275494
           5     0.9968    1.0000    0.9984    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9998    0.8788    0.9354     69052
           8     0.9953    1.0000    0.9977    157261
           9     0.6445    0.8333    0.7268      1962
          10     0.8108    0.9926    0.8925      1351
          11     0.4940    0.9570    0.6516     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9901   3294954
   macro avg     0.9168    0.9592    0.9299   3294954
weighted avg     0.9938    0.9901    0.9911   3294954



In [18]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start() 
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7171, Train Macro F1: 0.9717 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6258, Train Macro F1: 0.9983 | Val Loss: 0.5450, Val Macro F1: 0.9960


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6242, Train Macro F1: 0.9982 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6245, Train Macro F1: 0.9978 | Val Loss: 0.5494, Val Macro F1: 0.9955


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.6148, Train Macro F1: 0.9994 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.6143, Train Macro F1: 0.9996 | Val Loss: 0.5405, Val Macro F1: 0.9964


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.6140, Train Macro F1: 0.9996 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.6138, Train Macro F1: 0.9996 | Val Loss: 0.5404, Val Macro F1: 0.9965

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9994    0.9999    0.9997    246000
           1     0.9988    0.9998    0.9993    613218
           2     0.9782    0.8339    0.9004     92157
           3     1.0000    0.9999    1.0000    552847
           4     1.0000    0.9707    0.9851    275494
           5     0.9990    1.0000    0.9995    234000
           6     0.9995    1.0000    0.9997    689483
           7     0.9998    0.7869    0.8807     69052
           8     0.9966    1.0000    0.9983    157261
           9     0.7365    0.8318    0.7812      1962
          10     0.8117    0.9926    0.8931      1351
          11     0.4638    0.9740    0.6284     25973
          12     0.9840    1.0000    0.9919    336156

    accuracy                         0.9881   3294954
   macro avg     0.9206    0.9530    0.9275   3294954
weighted avg     0.9927    0.9881    0.9892   3294954



Undersampling + CNN + LSTM + Attention + Sliding Window

In [19]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [20]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [10]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [10]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giảm từ 49200 xuống 20000 cửa sổ.
Class 1: Giảm từ 122644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 18432 cửa sổ (step=5).
Class 3: Giảm từ 110569 xuống 20000 cửa sổ.
Class 4: Giảm từ 55098 xuống 20000 cửa sổ.
Class 5: Giảm từ 46800 xuống 20000 cửa sổ.
Class 6: Giảm từ 137896 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 13810 cửa sổ (step=5).
Class 8: Giảm từ 31452 xuống 20000 cửa sổ.
Class 9: Giữ nguyên 392 cửa sổ (step=5).
Class 10: Giữ nguyên 1349 cửa sổ (sử dụng step=1 để bảo toàn).
Class 11: Giữ nguyên 5194 cửa sổ (step=5).
Class 12: Giảm từ 67231 xuống 20000 cửa sổ.
Tổng số cửa sổ sau khi lọc và Undersampling: 199177
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 328000 cửa sổ (step=1).
Class 1: Giữ nguyên 817622 cửa sổ (step

In [22]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights

# model CNN-BiLSTM với Cơ chế Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256,10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # THAY ĐỔI DUY NHẤT Ở ĐÂY: in_channels = num_features (thay vì 1)
        # Conv1d sẽ trượt dọc theo chiều thời gian (Time_Steps) thay vì trượt dọc theo các features.
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: giữ nguyên
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: giữ nguyên
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: giữ nguyên
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention: giữ nguyên
        self.attention = Attention(128 * 2) 

        # các tầng dense: giữ nguyên
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng: giữ nguyên
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ SlidingWindowDataset đang có shape: (Batch, Time_Steps, Features)
        
        # THAY ĐỔI Ở ĐÂY: Dùng permute thay vì unsqueeze
        # Chuyển vị để shape trở thành: (Batch, Features, Time_Steps)
        # Để Conv1d quét qua chiều Time_Steps với số kênh tương ứng với số Features
        x = x.permute(0, 2, 1) 

        # đi qua 3 tầng Conv1dCNN (Code đoạn này giữ nguyên 100%)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM: (Batch, Channels, Time_Steps_New) -> (Batch, Time_Steps_New, Channels)
        # Chiều Channels lúc này đang là 128 (khớp hoàn hảo với input_size=128 của BiLSTM)
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out

In [24]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7406, Train Macro F1: 0.8426 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.5991, Train Macro F1: 0.9088 | Val Loss: 0.5547, Val Macro F1: 0.8839


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.5945, Train Macro F1: 0.9091 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.5919, Train Macro F1: 0.9091 | Val Loss: 0.5422, Val Macro F1: 0.8883


...

Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.5750, Train Macro F1: 0.9443 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.5750, Train Macro F1: 0.9652 | Val Loss: 0.5414, Val Macro F1: 0.9984

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     1.0000    0.9854    0.9927    246000
           1     0.9668    1.0000    0.9831    613218
           2     0.9387    0.8586    0.8969     92157
           3     1.0000    0.9998    0.9999    552847
           4     1.0000    0.9079    0.9517    275494
           5     0.9961    1.0000    0.9980    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9997    0.8095    0.8946     69052
           8     0.9574    1.0000    0.9782    157261
           9     0.0000    0.0000    0.0000      1962
          10     0.3932    0.9985    0.5642      1351
          11     0.5406    0.9776    0.6962     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9824   3294954
   macro avg     0.8302    0.8875    0.8427   3294954
weighted avg     0.9853    0.9824    0.9828   3294954



c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 0.7521, Train Macro F1: 0.8339 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6002, Train Macro F1: 0.9087 | Val Loss: 0.5446, Val Macro F1: 0.8881


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.5948, Train Macro F1: 0.9089 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.5915, Train Macro F1: 0.9092 | Val Loss: 0.6620, Val Macro F1: 0.8535


...

Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.5753, Train Macro F1: 0.9132 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.5754, Train Macro F1: 0.9132 | Val Loss: 0.5405, Val Macro F1: 0.8897

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     1.0000    0.9908    0.9954    246000
           1     0.9852    1.0000    0.9925    613218
           2     0.9565    0.8690    0.9106     92157
           3     1.0000    0.9996    0.9998    552847
           4     1.0000    0.9276    0.9624    275494
           5     0.9947    1.0000    0.9973    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9997    0.8000    0.8888     69052
           8     0.9304    1.0000    0.9639    157261
           9     0.0000    0.0000    0.0000      1962
          10     0.4076    0.9963    0.5786      1351
          11     0.5257    0.9787    0.6840     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9846   3294954
   macro avg     0.8307    0.8894    0.8441   3294954
weighted avg     0.9877    0.9846    0.9851   3294954



c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Undersampling + CNN + LSTM + Attention (attention + no sliding window) (cấu trúc giống paper gốc)

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 # trừ đi cột label
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# kiến trúc chính của paper
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # tầng 1: conv1d với 128 filters, kernel size = 5, padding = 2 để giũ nguyên chiều dài chuỗi
        # in_channels = 1 (Vì ta coi 314 features như 1 chuỗi dài 314 đơn vị) (đưa từng flow riêng lẻ vào)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: conv1d với 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: conv1d với 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: 128 units, Dropout(0.3)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # các tầng dense
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        # đầu vào x có shape: (256, 314)
        
        # shape trở thành: (256, 1, 314)
        x = x.unsqueeze(1) 

        # đi qua 3 tầng Conv1dCNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # không dùng softmax
        return out

# khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [4]:
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import DataLoader
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # nhóm theo nhãn và áp dụng undersampling
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # xáo trộn dữ liệu sau khi đã áp dụng undersampling để đảm bảo tính ngẫu nhiên
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# khởi tạo dataset với max_samples_per_class để giới hạn số lượng mẫu của mỗi class trong tập train, còn val và test thì giữ nguyên để đánh giá chính xác hơn
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df)
test_dataset = StandardFlowDataset(test_df) 

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 1.0245, Train Macro F1: 0.6793 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6710, Train Macro F1: 0.8951 | Val Loss: 0.5594, Val Macro F1: 0.9310


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6389, Train Macro F1: 0.9507 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6253, Train Macro F1: 0.9721 | Val Loss: 0.5558, Val Macro F1: 0.9544


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.5908, Train Macro F1: 0.9872 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.5911, Train Macro F1: 0.9862 | Val Loss: 0.5530, Val Macro F1: 0.9687


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.5905, Train Macro F1: 0.9868 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.5909, Train Macro F1: 0.9868 | Val Loss: 0.5498, Val Macro F1: 0.9740

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9930    0.9990    0.9960    246000
           1     0.9980    0.9972    0.9976    613218
           2     0.9210    0.8437    0.8807     92157
           3     1.0000    0.9945    0.9972    552847
           4     0.9976    0.9049    0.9490    275494
           5     0.9025    0.9994    0.9485    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9508    0.7028    0.8082     69052
           8     0.9971    0.9999    0.9985    157261
           9     0.6987    0.8451    0.7649      1962
          10     0.6856    0.9779    0.8061      1360
          11     0.4651    0.9467    0.6237     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9793   3294963
   macro avg     0.8930    0.9393    0.9054   3294963
weighted avg     0.9841    0.9793    0.9802   3294963



In [7]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    if epoch % 2 == 0:
        print(f"Epoch {epoch+1} là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.")
        val_macro_f1 = 0.0  
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {'N/A'}, Val Macro F1: {val_macro_f1:.4f}")
        continue
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [1/30] - Train Loss: 1.0210, Train Macro F1: 0.6833 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [2/30] - Train Loss: 0.6639, Train Macro F1: 0.8990 | Val Loss: 0.5649, Val Macro F1: 0.8721


Epoch 3 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [3/30] - Train Loss: 0.6378, Train Macro F1: 0.9093 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [4/30] - Train Loss: 0.6275, Train Macro F1: 0.9131 | Val Loss: 0.5585, Val Macro F1: 0.8731


...

Epoch 27 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [27/30] - Train Loss: 0.5886, Train Macro F1: 0.9872 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [28/30] - Train Loss: 0.5882, Train Macro F1: 0.9879 | Val Loss: 0.5500, Val Macro F1: 0.9711


Epoch 29 là epoch chẵn, bỏ qua đánh giá trên tập Val để tiết kiệm thời gian.
Epoch [29/30] - Train Loss: 0.5886, Train Macro F1: 0.9876 | Val Loss: N/A, Val Macro F1: 0.0000


Epoch [30/30] - Train Loss: 0.5885, Train Macro F1: 0.9880 | Val Loss: 0.5512, Val Macro F1: 0.9778

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9936    0.9990    0.9963    246000
           1     0.9588    0.9974    0.9777    613218
           2     0.9401    0.7869    0.8567     92157
           3     1.0000    0.9957    0.9979    552847
           4     0.9967    0.9088    0.9507    275494
           5     1.0000    0.9970    0.9985    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9990    0.6811    0.8100     69052
           8     0.9976    0.9986    0.9981    157261
           9     0.7867    0.8308    0.8081      1962
          10     0.7660    0.9963    0.8661      1360
          11     0.4034    0.9801    0.5716     25973
          12     0.9985    1.0000    0.9992    336156

    accuracy                         0.9779   3294963
   macro avg     0.9108    0.9363    0.9101   3294963
weighted avg     0.9847    0.9779    0.9793   3294963

